# 类型兼容性

学习目标：能按结构和调用方向判断可赋值性，并识别严格函数检查仍保留的边界。

前置知识：TypeScript 对象结构、联合、函数签名与回调；JavaScript 参数传递。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。本章附加选项：strictFunctionTypes=true，含义见对应知识点。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/08-type-compatibility/。

1. [main.ts](scripts/08-type-compatibility/main.ts)：按正文顺序组织的正常示例，片段依赖同文件前文定义。
2. [type-errors.ts](scripts/08-type-compatibility/type-errors.ts)：与正常示例隔离的类型反例，不生成或执行 JavaScript。
3. [tsconfig.json](scripts/08-type-compatibility/tsconfig.json)、[tsconfig.errors.json](scripts/08-type-compatibility/tsconfig.errors.json)：分别明确正常与反例文件范围。
4. [method-runtime-error.ts](scripts/08-type-compatibility/method-runtime-error.ts)：单独说明和执行的配套边界示例。



Step 1：检查正常项目的类型。

```bash
npm run check:08
```

Step 2：生成正常项目的 JavaScript。

```bash
npm run build:08
```

Step 3：运行正常示例。

```bash
npm run run:08
```

Step 4：检查下文独立列出的类型反例。

```bash
npm run errors:08
# 预期非零退出；按反例注释逐行核对具体错误，不运行 type-errors.ts。
```

正常配置只包含上面列出的正常与独立运行示例，生成文件位于 .build/08-type-compatibility/。错误配置继承正常选项，改用 type-errors.ts 并开启 noEmit。

## 1 结构类型与可赋值方向

先从目标列出必需字段，再回到来源逐项寻找，才能看清赋值方向。

结构类型系统（structural typing）主要按成员判断类型关系，不要求对象显式声明自己实现某个接口。把来源赋给目标时，来源必须提供目标要求的兼容成员；来源可以有更多成员。

可赋值性是有方向的：更多成员的对象可作为较少成员的视图，反方向缺少成员则不成立。嵌套成员也要兼容。这不是运行时复制、删字段或添加标记。类的 private、protected 等会增加限制，见类章节。

![对象赋值：来源要满足目标要求。只讨论本例的普通结构类型；赋值不会删除对象的额外属性。](image/illustration/08-01-structural-assignability.svg)

图示说明：图中的箭头是静态可赋值方向，不是对象复制或字段投影。

下面从 detailed 赋给 named 后，再检查 minutes 是否仍存在；反例则检查目标要求的字段缺在哪里。

```typescript
export {};
type Named = { title: string };
type Timed = { title: string; minutes: number };
const detailed: Timed = { title: "兼容性", minutes: 20 };
const named: Named = detailed;
function readTitle(value: Named): string { return value.title; }
console.log(readTitle(detailed), "minutes" in named); // 兼容性 true
```

以下片段来自独立的 type-errors.ts：

```typescript
type NeedBoth = { title: string; minutes: number };
const onlyTitle = { title: "兼容性" };
const insufficient: NeedBoth = onlyTitle; // 缺少目标必需成员 minutes。
const nested: { meta: { count: number } } = { meta: { count: "1" } }; // 嵌套成员不兼容。
```

## 2 对象字面量的额外属性检查

新鲜对象字面量在带目标类型的赋值或传参位置受到额外属性检查，以发现容易被忽略的拼写与配置错误。经过变量的结构兼容和直接字面量检查，是两个相关但不同的检查层面。

当需求确实允许额外成员时，应描述正确接口或索引签名；当只是把较丰富对象交给只读某些成员的函数时，结构兼容就足够。把字面量先塞进变量不该成为规避真实错误的通用办法。

```typescript
const full = { title: "笔记", author: "林" };
const titleView: Named = full;
console.log(titleView.title); // 笔记
```

以下片段来自独立的 type-errors.ts：

```typescript
const fresh: { title: string } = { title: "笔记", author: "林" }; // 新鲜字面量存在额外 author。
```

## 3 函数参数逆向、返回值同向

把一次调用分成输入与返回两段，比只记“逆变、协变”更容易判断替换是否安全。

在本章显式启用 strictFunctionTypes 的配置下，普通函数类型的参数必须能接收目标可能提供的输入。接收 string | number 的函数能替代只接收 string 的位置；只接收 string 的函数不能替代可能收到数值的位置。这称为参数的逆变方向。

返回值按相反方向检查：提供者的结果应包含调用者要求的结构。返回带 title 和 minutes 的对象可以满足只要求 title 的函数类型；反过来不成立。void 的返回值忽略规则是函数章节中的特殊情况。

![函数替换：输入要接得住，输出要给得够。strictFunctionTypes 下的普通函数类型；方法参数例外不包含在此图中。](image/illustration/08-02-function-variance.svg)

图示说明：图不讨论 void 特例或方法双向检查；这些边界保留在对应小节。

先用 textOnly 的可能输入检查 general，再用 produceDetailed 的结果检查 produceNamed；随后反向尝试两个错误例子。

```typescript
const general = (value: string | number): string => String(value);
const textOnly: (value: string) => string = general;
const produceDetailed = (): Timed => ({ title: "函数", minutes: 12 });
const produceNamed: () => Named = produceDetailed;
console.log(textOnly("TS"), produceNamed().title); // TS 函数
```

以下片段来自独立的 type-errors.ts：

```typescript
const narrowInput = (value: string): string => value.toUpperCase();
const unsafeInput: (value: string | number) => string = narrowInput; // 不能接收目标可能传来的 number。
const basicResult = () => ({ title: "函数" });
const missingReturn: () => { title: string; minutes: number } = basicResult; // 返回值缺少 minutes。
```

## 4 参数数量、可选与剩余参数

函数实现可以忽略调用者提供的额外参数，因此只接收一个参数的函数可以替代会收到两个参数的位置。反过来，不能让需要两个必需参数的函数替代只保证提供一个参数的位置。

可选参数表示调用方可以不给；严格空值检查下，实现就必须能处理 undefined。普通数组剩余参数在兼容比较中具有可选参数序列的效果，但不会保证任意索引存在。参数个数放宽与运行时数组索引风险可以同时出现；显式的必需元组位置能表达更强约束。

```typescript
const useFirst = (value: number): string => String(value);
const twoInputs: (value: number, label: string) => string = useFirst;
const optional = (value?: number): string => String(value ?? 0);
const required: (value: number) => string = optional;
const rest = (...values: number[]): string => values.join(",");
const acceptsPair: (left: number, right: number) => string = rest;
console.log(twoInputs(7, "忽略"), required(0), acceptsPair(1, 2)); // 7 0 1,2
```

以下片段来自独立的 type-errors.ts：

```typescript
const twoRequired = (left: number, right: number): number => left + right;
const oneSlot: (left: number) => number = twoRequired; // 调用方不保证提供 right。
const requiredValue = (value: number): number => value;
const optionalSlot: (value?: number) => number = requiredValue; // 目标允许不给参数。
```

## 5 方法参数的双向检查例外

strictFunctionTypes 不把相同的严格参数规则施加到方法语法。方法参数可按双向可赋值关系比较（bivariance），以兼容既有类层次，但它可能放行不安全替换。

下面两个对象的方法形参范围不同，赋值可以通过。正常示例仅传入字符串；独立反例给相同视图传入数值，最终调用字符串方法而失败。函数属性写法 handler: (value: ...) =&gt; ... 与方法写法 handler(value: ...): ... 的边界必须分清。

```typescript
interface MethodView { handle(value: string | number): string; }
const specialized = { handle(value: string): string { return value.toUpperCase(); } };
const methodView: MethodView = specialized;
console.log(methodView.handle("ts")); // TS
```

以下片段来自独立的 type-errors.ts：

```typescript
interface FunctionView { handle: (value: string | number) => string; }
const stringHandler = { handle: (value: string): string => value.toUpperCase() };
const saferView: FunctionView = stringHandler; // 函数属性受严格参数检查。
```

## 6 健全性边界的实际观察

类型系统的健全性（soundness）涉及被允许的操作能否确保类型相关的运行行为安全。TypeScript 明确保留部分不能在编译时保证安全的操作，所以 strict 也不是所有运行时错误的证明。

以下是 method-runtime-error.ts 的完整反例，类型检查放行的是方法兼容性；异常来自实际的数值输入。不要通过关闭 strictFunctionTypes 或加断言修复，应让实现处理所有允许输入，或收紧调用契约。

```typescript
export {};
interface MethodView { handle(value: string | number): string; }
const specialized = { handle(value: string): string { return value.toUpperCase(); } };
const methodView: MethodView = specialized;
methodView.handle(3); // TypeError：方法兼容性没有让 number 获得字符串方法。
```

Step 1：在已执行 build:08 后单独运行反例。

```bash
node .build/08-type-compatibility/method-runtime-error.js
# 预期退出码 1；TypeError，消息包含 value.toUpperCase is not a function。
```

## 本章小结

结构兼容检查来源是否满足目标；函数的输入与输出方向不同。字面量检查、可选参数和方法参数例外均有具体适用位置，strict 仍需配合真实输入边界与运行检查。

## 练习

1. 画出能接受 string | number 的回调和只能接受 string 的回调的赋值方向，并用两次赋值核对：安全方向通过，危险方向失败。
2. 将反例方法改为函数属性，确认危险赋值变成类型错误；再实现能处理联合输入的函数，两个输入都正常返回。
3. 写一个要求首个数值必有、其后任意多个数值的元组剩余参数函数；空调用应失败，一个或多个数值应通过并运行。

## 参考与引用来源

- TypeScript 官方文档：[Type Compatibility：结构、soundness、函数、可选与剩余参数](https://www.typescriptlang.org/docs/handbook/type-compatibility.html)；[strictFunctionTypes：方法例外](https://www.typescriptlang.org/tsconfig/strictFunctionTypes.html)；[2.6：逆变与方法双变](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-2-6.html#strict-function-types)；[Object Types：额外属性检查](https://www.typescriptlang.org/docs/handbook/2/objects.html#excess-property-checks)。

- npm 官方文档：[npm run（v11）](https://docs.npmjs.com/cli/v11/commands/npm-run/)：从本技术目录运行已配置脚本，并解析本地工具。